# Laboratory Task 6

**Francis Rey C. Torillo**

In [35]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [36]:
!pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable


In [37]:
import torch
import numpy as np
import random
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split


In [38]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        # First convolutional layer: 1 input channel (grayscale), 16 output channels, kernel size 3x3, padding 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Second convolutional layer: 16 input channels, 32 output channels, kernel size 3x3, padding 1
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        # MNIST images are 28x28. After two 2x2 max-pooling layers, size becomes 7x7.
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        # Conv block 1
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool1(x)
        
        # Conv block 2
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool2(x)
        
        # Flatten for fully connected layers
        x = x.view(-1, 32 * 7 * 7)
        
        # FC layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        # Softmax activation on the output layer
        return F.softmax(x, dim=1)

    def set_seed(seed=143):
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)
        random.seed(seed)
        torch.backends.cudnn.deterministic = True


set_seed(143)
model = CNNModel()
print(model)

CNNModel(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [39]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [40]:
# Define standard normalization transforms for MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download and load the MNIST dataset
full_train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Split training data into training (50,000) and validation (10,000) sets
train_data, val_data = random_split(full_train_data, [50000, 10000])

# Create the DataLoaders
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [41]:
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    # Validation phase
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {running_loss / len(train_loader):.4f} | "
          f"Val Loss: {val_loss / len(val_loader):.4f} | "
          f"Val Accuracy: {100 * correct / total:.2f}%")

Epoch 1/3: 100%|██████████| 782/782 [00:54<00:00, 14.31it/s]


Epoch [1/3] | Train Loss: 1.6263 | Val Loss: 1.5745 | Val Accuracy: 88.70%


Epoch 2/3: 100%|██████████| 782/782 [00:51<00:00, 15.28it/s]


Epoch [2/3] | Train Loss: 1.5029 | Val Loss: 1.4869 | Val Accuracy: 97.64%


Epoch 3/3: 100%|██████████| 782/782 [00:51<00:00, 15.21it/s]


Epoch [3/3] | Train Loss: 1.4811 | Val Loss: 1.4797 | Val Accuracy: 98.19%


In [42]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

# Calculate metrics imported in Cell 2
test_accuracy = accuracy_score(all_labels, all_preds)
conf_matrix = confusion_matrix(all_labels, all_preds)
class_report = classification_report(all_labels, all_preds)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%\n")
print("Classification Report:\n", class_report)

Test Accuracy: 98.31%

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.98       980
           1       0.99      0.99      0.99      1135
           2       0.97      0.99      0.98      1032
           3       0.99      0.97      0.98      1010
           4       1.00      0.98      0.99       982
           5       0.97      0.99      0.98       892
           6       0.99      0.98      0.99       958
           7       0.97      0.99      0.98      1028
           8       0.99      0.98      0.98       974
           9       0.99      0.97      0.98      1009

    accuracy                           0.98     10000
   macro avg       0.98      0.98      0.98     10000
weighted avg       0.98      0.98      0.98     10000

